In [2]:
!pip install networkx numpy


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import json
import os
import networkx as nx
import numpy as np
from datetime import datetime

cluster_summary_path = "../d1/results/cluster_summary.json"
with open(cluster_summary_path, 'r', encoding='utf-8') as f:
    cluster_summary = json.load(f)

services_path = "dataset/services.json"
with open(services_path, 'r', encoding='utf-8') as f:
    services_data = json.load(f)

incidents_path = "dataset/incidents_history.json"
with open(incidents_path, 'r', encoding='utf-8') as f:
    incidents_data = json.load(f)

alerts_path = "dataset/alerts_sample.jsonl"
alerts = []
with open(alerts_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            alerts.append(json.loads(line))

print(f"Loaded {len(cluster_summary['clusters'])} clusters.")
print(f"Loaded {len(services_data['services'])} services, {len(services_data['edges'])} edges.")
print(f"Loaded {len(incidents_data['incidents'])} historical incidents.")
print(f"Loaded {len(alerts)} alerts.")

Loaded 1 clusters.
Loaded 10 services, 17 edges.
Loaded 29 historical incidents.
Loaded 20 alerts.


In [4]:
# Khởi tạo đồ thị định hướng
G = nx.DiGraph()

# TODO: Duyệt qua services_data['services'] và thêm node vào G
# Ví dụ: G.add_node(s['name'], type='service', tier=s.get('tier'), criticality=s.get('criticality'))
for s in services_data['services']:
    G.add_node(s['name'], type='service', tier=s.get('tier'), criticality=s.get('criticality'))

# TODO: Duyệt qua services_data['stores'] và thêm node vào G
for st in services_data['stores']:
    G.add_node(st['name'], type=st.get('type'), criticality=st.get('criticality'))

# TODO: Duyệt qua services_data['edges'] và thêm edge vào G
for e in services_data['edges']:
    G.add_edge(e['from'], e['to'], type=e.get('type'))

print(f"Đồ thị có {G.number_of_nodes()} nodes và {G.number_of_edges()} edges.")
assert nx.is_directed(G)


Đồ thị có 14 nodes và 17 edges.


In [5]:
def calculate_graph_temporal_candidates(cluster, alerts, G):
    cluster_services = cluster['services']
    
    # 1. Tạo subgraph từ các service trong cluster
    subgraph = G.subgraph(cluster_services)
    
    if len(subgraph) == 0:
        return []
    
    # 2. Tính PageRank trên reverse graph của subgraph
    try:
        rev_subgraph = subgraph.reverse(copy=True)
        pagerank_scores = nx.pagerank(rev_subgraph, alpha=0.85)
    except Exception:
        # Fallback nếu PageRank lỗi (đồ thị rỗng, không hội tụ...)
        pagerank_scores = {node: 1.0/len(subgraph) for node in subgraph.nodes()}

    # Normalize PageRank scores về khoảng [0, 1] (chia cho max score trong cluster)
    max_pr = max(pagerank_scores.values()) if pagerank_scores else 1.0
    pagerank_norm = {node: (score / max_pr if max_pr > 0 else 0.0) for node, score in pagerank_scores.items()}
    
    # 3. Tính Temporal Score
    cluster_alert_ids = set(cluster['alert_ids'])
    cluster_alerts = [a for a in alerts if a['id'] in cluster_alert_ids]
    
    # Lấy timestamp sớm nhất của mỗi service trong cluster
    service_earliest_ts = {}
    for a in cluster_alerts:
        svc = a['service']
        ts = datetime.fromisoformat(a['ts'].replace('Z', '+00:00'))
        if svc not in service_earliest_ts or ts < service_earliest_ts[svc]:
            service_earliest_ts[svc] = ts
            
    # Tìm min_ts và max_ts
    if service_earliest_ts:
        timestamps = list(service_earliest_ts.values())
        min_ts = min(timestamps)
        max_ts = max(timestamps)
        time_diff = (max_ts - min_ts).total_seconds()
    else:
        time_diff = 0
        
    #Tính temporal score cho mỗi service: sớm nhất = 1.0, muộn nhất = 0.0
    temporal_scores = {}
    for svc in cluster_services:
        if svc not in service_earliest_ts:
            temporal_scores[svc] = 0.0
            continue
        if time_diff == 0:
            temporal_scores[svc] = 1.0
        else:
            svc_ts = service_earliest_ts[svc]
            delay = (svc_ts - min_ts).total_seconds()
            temporal_scores[svc] = 1.0 - (delay / time_diff)
            
    #  Kết hợp theo công thức: 0.6 * pagerank_norm + 0.4 * temporal_score
    candidates = []
    for svc in cluster_services:
        pr = pagerank_norm.get(svc, 0.0)
        temp = temporal_scores.get(svc, 0.0)
        final_score = 0.6 * pr + 0.4 * temp
        candidates.append((svc, final_score))
        
    # Sắp xếp giảm dần theo score
    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates

first_cluster = cluster_summary['clusters'][0]
candidates = calculate_graph_temporal_candidates(first_cluster, alerts, G)
print("Candidates cho cluster 0:", candidates)

Candidates cho cluster 0: [('edge-lb', 0.8975778546712803), ('checkout-svc', 0.7766998577168194), ('payment-svc', 0.5232673510935538), ('cart-svc', 0.39731579400012823), ('notification-svc', 0.3724022991904396), ('recommender-svc', 0.26167565559182365), ('search-svc', 0.12326735109355379)]


In [6]:
def retrieve_similar_incidents(cluster, incidents, k=3):
    cluster_services = set(cluster['services'])
    max_severity = cluster.get('max_severity', 'warn').lower()
    
    scored_incidents = []
    
    for inc in incidents:
        score = 0.0
        
        # Cập nhật điểm +0.4 nếu inc['root_cause_service'] nằm trong cluster_services
        if inc.get('root_cause_service') in cluster_services:
            score += 0.4
            
        # Cập nhật điểm +0.2 cho mỗi service overlap giữa cluster_services và inc['services_involved'] (tối đa +0.4)
        inc_services = set(inc.get('services_involved', []))
        overlap = cluster_services.intersection(inc_services)
        overlap_score = 0.2 * len(overlap)
        score += min(overlap_score, 0.4)
        
        # Cập nhật điểm +0.2 nếu inc['severity'].lower() == max_severity
        if inc.get('severity', '').lower() == max_severity:
            score += 0.2
            
        scored_incidents.append((inc, score))
        
    # Sắp xếp giảm dần theo similarity score
    scored_incidents.sort(key=lambda x: x[1], reverse=True)
    
    # Lấy top-k incidents có similarity score >= 0.2
    filtered_incidents = [(inc['id'], score) for inc, score in scored_incidents if score >= 0.2]
    return filtered_incidents[:k]

similar_incidents = retrieve_similar_incidents(first_cluster, incidents_data['incidents'])
print("Similar incidents cho cluster 0:", similar_incidents)

Similar incidents cho cluster 0: [('INC-2025-11-08', 0.8), ('INC-2026-03-20', 0.8), ('INC-2025-08-02', 0.6000000000000001)]


In [7]:
def classify_rca(cluster, candidates, similar_incidents, incidents_dict):
    # fallback nếu không tìm thấy incident lịch sử nào tương đồng
    if not similar_incidents or similar_incidents[0][1] < 0.2:
        top_candidate = candidates[0][0] if candidates else "unknown"
        confidence = candidates[0][1] if candidates else 0.5
        return {
            "root_cause": top_candidate,
            "class": "other",
            "confidence": round(float(confidence), 2),
            "actions": ["Investigate manually"],
            "reasoning": "No similar past incidents found. Fallback to top candidate from graph topology and temporal analysis.",
            "similar_incidents": [],
            "method": "graph-only-fallback"
        }
        
    top_inc_id, similarity = similar_incidents[0]
    top_inc = incidents_dict[top_inc_id]
    
    root_cause_service = top_inc['root_cause_service']
    root_cause_class = top_inc['root_cause_class']
    remediation = top_inc['remediation']
    
    # Hallucination Guard: Đảm bảo root cause nằm trong cluster
    if root_cause_service not in cluster['services']:
        root_cause = candidates[0][0] if candidates else "unknown"
    else:
        root_cause = root_cause_service
        
    return {
        "root_cause": root_cause,
        "class": root_cause_class,
        "confidence": round(float(similarity), 2),
        "actions": [remediation] if isinstance(remediation, str) else remediation,
        "reasoning": f"Matched with historical incident {top_inc_id} (similarity: {similarity:.2f}). Summary: {top_inc['summary']}",
        "similar_incidents": [inc_id for inc_id, _ in similar_incidents],
        "method": "graph+retrieval"
    }

# Tạo lookup dict
inc_lookup = {inc['id']: inc for inc in incidents_data['incidents']}

rca_res = classify_rca(first_cluster, candidates, similar_incidents, inc_lookup)
print("RCA result cho cluster 0:\n", json.dumps(rca_res, indent=2))

RCA result cho cluster 0:
 {
  "root_cause": "payment-svc",
  "class": "connection_pool_exhaustion",
  "confidence": 0.8,
  "actions": [
    "Rollback to v3.1. Scale pool 50 \u2192 100 cushion. Add pool monitor alert > 80%."
  ],
  "reasoning": "Matched with historical incident INC-2025-11-08 (similarity: 0.80). Summary: Payment-svc v3.2 deploy at 09:42 leak DB pool. Pool 50/50 used trong 5 ph\u00fat. Downstream checkout cascade. Notification queue backed up.",
  "similar_incidents": [
    "INC-2025-11-08",
    "INC-2026-03-20",
    "INC-2025-08-02"
  ],
  "method": "graph+retrieval"
}


In [11]:

output_results = []

for cluster in cluster_summary['clusters']:
    c_id = cluster['cluster_id']
    
    # Tính graph candidates
    candidates = calculate_graph_temporal_candidates(cluster, alerts, G)
    
    # Tìm incidents tương tự
    similar = retrieve_similar_incidents(cluster, incidents_data['incidents'])
    
    # Phân loại RCA
    rca_info = classify_rca(cluster, candidates, similar, inc_lookup)
    
    # Chuẩn hóa format list graph_top3 (mỗi phần tử là [service, score])
    graph_top3 = [[svc, round(score, 2)] for svc, score in candidates[:3]]
    
    res_obj = {
        "cluster_id": c_id,
        "graph_top3": graph_top3,
        "root_cause": rca_info["root_cause"],
        "class": rca_info["class"],
        "confidence": rca_info["confidence"],
        "actions": rca_info["actions"],
        "reasoning": rca_info["reasoning"],
        "similar_incidents": rca_info["similar_incidents"],
        "method": rca_info["method"]
    }
    output_results.append(res_obj)

final_output = {
    "clusters_analyzed": len(cluster_summary['clusters']),
    "results": output_results
}

output_file_path = "rca_output.json"
with open(output_file_path, 'w', encoding='utf-8') as f:
    json.dump(final_output, f, indent=2, ensure_ascii=False)

print(f"Successfully wrote RCA output to {output_file_path}")

Successfully wrote RCA output to rca_output.json
